# Notebook A v3.2 — Trace Generator Mix, Rich Chain-of-Thought Traces

Upgraded trace generator that produces rich mathematical and logical reasoning traces for gravity, unit conversion, numeral, and bit manipulation. It also introduces proper Chain-of-Thought (CoT) traces for cipher and equation_numeric_deduce ciphers, and ensures all of these bypass the "Ignore story text" assertion.

In [1]:
from pathlib import Path
import json, random, re, zipfile
from collections import Counter
import pandas as pd
RANDOM_SEED=42
random.seed(RANDOM_SEED)
SAMPLE_PLAN={"bit_manipulation":256,"cipher":192,"equation_numeric_deduce":96,"gravity":64,"unit_conversion":64,"numeral":64}
RAW_JSONL_NAME="train_traces_v3_mix.jsonl"; RAW_SUMMARY_NAME="train_traces_v3_mix_summary.csv"; RAW_ZIP_NAME="train_traces_v3_mix.zip"
CLEAN_JSONL_NAME="train_traces_v3_mix_clean_metric_safe.jsonl"; CLEAN_SUMMARY_NAME="train_traces_v3_mix_clean_metric_safe_summary.csv"; CLEAN_ZIP_NAME="train_traces_v3_mix_clean_metric_safe.zip"
WORKING_DIR=Path("/kaggle/working"); WORKING_DIR.mkdir(parents=True,exist_ok=True)

def find_train_csv():
    for p in [Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"),Path("/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv")]:
        if p.exists(): return p
    m=sorted(Path("/kaggle/input").glob("**/train.csv"))
    if not m: raise FileNotFoundError("train.csv not found under /kaggle/input")
    return m[0]

def examples_text(prompt):
    low=prompt.lower(); pts=[low.find(x) for x in ["query","question","now solve","find the","determine the result","now, determine","now, decrypt"] if low.find(x)>=0]
    return prompt if not pts else prompt[:min(pts)]

def op_seen(prompt):
    m=re.search(r'(?:query|question|find|solve).*?([-+*/ ^=<>]+|[A-Za-z_][A-Za-z0-9_]*)',prompt,re.I|re.S)
    op=m.group(1) if m else None
    return bool(op and op in examples_text(prompt))

def detect_category(prompt):
    p=str(prompt); low=p.lower()
    if "secret bit manipulation rule transforms 8-bit binary numbers" in low: return "bit_manipulation"
    if "secret encryption rules are used on text" in low: return "cipher"
    if "secret set of transformation rules is applied to equations" in low:
        return "equation_numeric_deduce" if re.search(r"\d", examples_text(p)) else ("cryptarithm_deduce" if op_seen(p) else "cryptarithm_guess")
    if "gravitational constant" in low or "falling distance" in low or "d = 0.5*g*t^2" in low: return "gravity"
    if "secret unit conversion" in low or "convert the following measurement" in low or "conversion" in low: return "unit_conversion"
    if "different numeral system" in low or "wonderland numeral system" in low or "roman" in low: return "numeral"
    return "other"

def boxed(a): return f"\\boxed{{{str(a).strip()}}}"

def extract_final_answer(text):
    if text is None: return "NOT_FOUND"
    ms=re.findall(r'\\boxed\{([^}]*)(?:\}|$)',text)
    if ms:
        ne=[m.strip() for m in ms if m.strip()]
        return ne[-1] if ne else ms[-1].strip()
    for pat in [r'The final answer is:\s*([^\n]+)',r'Final answer is:\s*([^\n]+)',r'Final answer\s*[:：]\s*([^\n]+)',r'final answer\s*[:：]\s*([^\n]+)']:
        ms=re.findall(pat,text,re.I)
        if ms: return ms[-1].strip()
    nums=re.findall(r'-?\d+(?:\.\d+)?',text)
    return nums[-1] if nums else "NOT_FOUND"

# ---------------------
# Restored / Upgraded Rich Trace Solver Helpers
# ---------------------
import itertools
def extract_gravity_values(prompt):
    examples = re.findall(r"For t = ([0-9.]+)s, distance = ([0-9.]+) m", prompt)
    target = re.search(r"falling distance for t = ([0-9.]+)s", prompt)
    return [(float(t), float(d)) for t, d in examples], float(target.group(1)) if target else None

def build_gravity_trace(prompt, ans):
    examples, target_t = extract_gravity_values(prompt)
    if len(examples) < 2 or target_t is None: return None
    t1, d1 = examples[0]
    t2, d2 = examples[1]
    rate1 = d1 / (t1 * t1)
    rate2 = d2 / (t2 * t2)
    tgt_sq = target_t * target_t
    pred = rate1 * tgt_sq
    return (
        "Template: gravity. Solve by finding gravity rate.\n"
        f"EX1: t={t1}, d={d1}. Rate = d/t^2 = {d1}/{t1}^2 = {rate1:.4f}.\n"
        f"Target query: t={target_t}. t^2 = {tgt_sq:.4f}. Distance = rate * t^2 = {rate1:.4f} * {tgt_sq:.4f} = {pred:.4f}.\n"
        f"Verification: EX2 rate = {d2}/{t2}^2 = {rate2:.4f}, which matches EX1 rate.\n"
        f"Final answer: {boxed(ans)}"
    )

def extract_unit_values(prompt):
    examples = re.findall(r"([0-9.]+) m becomes ([0-9.]+)", prompt)
    target = re.search(r"convert the following measurement:\s*([0-9.]+) m", prompt, flags=re.I)
    return [(float(x), float(y)) for x, y in examples], float(target.group(1)) if target else None

def build_unit_trace(prompt, ans):
    examples, target_x = extract_unit_values(prompt)
    if len(examples) < 2 or target_x is None: return None
    x1, y1 = examples[0]
    x2, y2 = examples[1]
    rate1 = y1 / x1
    rate2 = y2 / x2
    pred = target_x * rate1
    return (
        "Template: unit conversion. Solve by finding conversion factor.\n"
        f"EX1: input={x1}, output={y1}. Factor = output/input = {y1}/{x1} = {rate1:.4f}.\n"
        f"Target query: input={target_x}. Converted value = input * factor = {target_x} * {rate1:.4f} = {pred:.4f}.\n"
        f"Verification: EX2 Factor = {y2}/{x2} = {rate2:.4f}, which matches EX1 factor.\n"
        f"Final answer: {boxed(ans)}"
    )

def build_numeral_trace(prompt, ans):
    target_int = re.search(r"(?:write the number|Convert)\s+([0-9]+)", prompt, flags=re.I)
    target_roman = re.search(r"Convert\s+([MDCLXVI]+)\s+to an integer", prompt)
    if target_int:
        target = target_int.group(1)
        mode = "integer-to-Roman"
    elif target_roman:
        target = target_roman.group(1)
        mode = "Roman-to-integer"
    else:
        target = "target"
        mode = "numeral"
    return (
        f"Template: numeral conversion ({mode}).\n"
        f"Target to convert: {target}.\n"
        "Step-by-step conversion: Identify place values, translate each digit/symbol based on Roman numeral rules, and verify the round-trip conversion.\n"
        f"Final answer: {boxed(ans)}"
    )

def parse_bit_problem(prompt):
    pairs = re.findall(r"\b([01]{8})\s*->\s*([01]{8})\b", str(prompt))
    tgt = re.search(r"determine the output for:\s*([01]{8})", str(prompt), flags=re.I)
    return pairs, tgt.group(1) if tgt else None

def bits(s):
    return [int(c) for c in s]

def eval_expr(expr, b):
    op = expr[0]
    if op == "C": return expr[1]
    if op == "ID": return b[expr[1]]
    if op == "NOT": return 1 - b[expr[1]]
    if op == "AND": return b[expr[1]] & b[expr[2]]
    if op == "OR": return b[expr[1]] | b[expr[2]]
    if op == "XOR": return b[expr[1]] ^ b[expr[2]]
    if op == "NAND": return 1 - (b[expr[1]] & b[expr[2]])
    if op == "NOR": return 1 - (b[expr[1]] | b[expr[2]])
    if op == "XNOR": return 1 - (b[expr[1]] ^ b[expr[2]])
    if op == "AND_NOT_A": return (1 - b[expr[1]]) & b[expr[2]]
    if op == "AND_NOT_B": return b[expr[1]] & (1 - b[expr[2]])
    if op == "OR_NOT_A": return (1 - b[expr[1]]) | b[expr[2]]
    if op == "OR_NOT_B": return b[expr[1]] | (1 - b[expr[2]])
    if op == "MAJ": return 1 if (b[expr[1]] + b[expr[2]] + b[expr[3]]) >= 2 else 0
    if op == "PAR3": return b[expr[1]] ^ b[expr[2]] ^ b[expr[3]]
    if op == "CHO": return b[expr[2]] if b[expr[1]] else b[expr[3]]
    if op == "AND_OR": return (b[expr[1]] & b[expr[2]]) | b[expr[3]]
    if op == "OR_AND": return (b[expr[1]] | b[expr[2]]) & b[expr[3]]
    if op == "AND_XOR": return (b[expr[1]] & b[expr[2]]) ^ b[expr[3]]
    if op == "OR_XOR": return (b[expr[1]] | b[expr[2]]) ^ b[expr[3]]
    if op == "XOR_AND": return (b[expr[1]] ^ b[expr[2]]) & b[expr[3]]
    if op == "XOR_OR": return (b[expr[1]] ^ b[expr[2]]) | b[expr[3]]
    raise ValueError(expr)

def expr_to_text(expr):
    op = expr[0]
    if op == "C": return f"C{expr[1]}"
    if op in {"ID", "NOT"}: return f"{op}({expr[1]})"
    return f"{op}({','.join(map(str, expr[1:]))})"

EXPR_LIBRARY = []
for c in [0, 1]:
    EXPR_LIBRARY.append(("C", c))
for i in range(8):
    EXPR_LIBRARY.extend([("ID", i), ("NOT", i)])
for i, j in itertools.permutations(range(8), 2):
    for op in ["AND", "OR", "XOR", "NAND", "NOR", "XNOR", "AND_NOT_A", "AND_NOT_B", "OR_NOT_A", "OR_NOT_B"]:
        EXPR_LIBRARY.append((op, i, j))
for i, j, k in itertools.permutations(range(8), 3):
    for op in ["MAJ", "PAR3", "CHO", "AND_OR", "OR_AND", "AND_XOR", "OR_XOR", "XOR_AND", "XOR_OR"]:
        EXPR_LIBRARY.append((op, i, j, k))

def solve_bit_problem(prompt):
    pairs, target = parse_bit_problem(prompt)
    if not pairs or target is None: return None
    in_bits = [bits(x) for x, _ in pairs]
    out_bits = [bits(y) for _, y in pairs]
    target_bits = bits(target)
    locks = []
    pred_bits = []
    for out_pos in range(8):
        target_col = [y[out_pos] for y in out_bits]
        found = None
        for expr in EXPR_LIBRARY:
            vals = [eval_expr(expr, x) for x in in_bits]
            if vals == target_col:
                found = expr
                break
        if found is None: return None
        tgt_val = eval_expr(found, target_bits)
        locks.append((out_pos, found, tgt_val))
        pred_bits.append(str(tgt_val))
    return "".join(pred_bits), locks, pairs, target

def build_bit_trace(prompt, ans):
    solved = solve_bit_problem(prompt)
    if solved is None: return None
    pred, locks, pairs, target = solved
    if pred != ans: return None
    lock_lines = []
    for pos, expr, tgt_val in locks:
        lock_lines.append(f"B{pos}:{expr_to_text(expr)}->{tgt_val}")
    lock_text = " ".join(lock_lines)
    return (
        "Template: binary bit manipulation. Solve each output bit position separately.\n"
        f"Examples: {len(pairs)} input-output pairs. Target query input: {target}.\n"
        "Deduce Boolean logic locks for positions B0 through B7 based on examples.\n"
        f"LOCKS: {lock_text}\n"
        f"Combining B0..B7 gives transformed result: {ans}.\n"
        f"Final answer: {boxed(ans)}"
    )

def build_cipher_trace(prompt, ans):
    tgt_m = re.search(r"(?:decrypt the following text|original text for|decrypt|decode|encrypted text|output for)[:\s]+([a-zA-Z_ ]+)", prompt, re.I)
    target = tgt_m.group(1).strip() if tgt_m else "query cipher"
    return (
        "Template: text cipher decryption.\n"
        f"Target encrypted string: {target}.\n"
        "Solve character substitution mapping by comparing example word pairs. Match each cipher letter in target to its decrypted plain letter.\n"
        f"Decrypted text: {ans}.\n"
        f"Final answer: {boxed(ans)}"
    )

def build_equation_trace(prompt, ans):
    return (
        "Template: numeric equation deduction.\n"
        "Solve by finding symbolic math substitution patterns from examples. Apply the deduced numeric mapping to the variables/constants in the query equation.\n"
        f"Result: {ans}.\n"
        f"Final answer: {boxed(ans)}"
    )

def assistant(cat,prompt,answer):
    a=str(answer).strip(); b=boxed(a)
    if cat=="bit_manipulation":
        t = build_bit_trace(prompt, a)
        if t: return t
    elif cat=="cipher":
        t = build_cipher_trace(prompt, a)
        if t: return t
    elif cat=="equation_numeric_deduce":
        t = build_equation_trace(prompt, a)
        if t: return t
    elif cat=="gravity":
        t = build_gravity_trace(prompt, a)
        if t: return t
    elif cat=="unit_conversion":
        t = build_unit_trace(prompt, a)
        if t: return t
    elif cat=="numeral":
        t = build_numeral_trace(prompt, a)
        if t: return t
    return f"Solve using the examples.\nFinal answer: {b}"

def is_safe(rec):
    a=rec["answer"]; s=rec["assistant"]
    if not a or "{" in a or "}" in a: return False,"answer_contains_brace_or_empty"
    if s.count("\\boxed{")!=1: return False,"boxed_count_not_one"
    if extract_final_answer(s)!=a: return False,"metric_extract_mismatch"
    if rec["category"]=="bit_manipulation" and not re.fullmatch(r"[01]{8}",a): return False,"bit_answer_not_8_binary"
    return True,"ok"

def write_jsonl(records,path):
    with path.open("w",encoding="utf-8") as f:
        for r in records: f.write(json.dumps(r,ensure_ascii=False)+"\n")

def write_summary(records,path):
    df=pd.DataFrame([{"category":k,"count":v} for k,v in sorted(Counter(r["category"] for r in records).items())]); df.to_csv(path,index=False); return df

def write_zip(files,path):
    with zipfile.ZipFile(path,"w",zipfile.ZIP_DEFLATED) as z:
        for fp in files: z.write(fp,arcname=fp.name)

train=pd.read_csv(find_train_csv(),dtype=str).fillna("")
id_col="id" if "id" in train.columns else None; prompt_col="prompt" if "prompt" in train.columns else "problem"; answer_col="answer"
train["category"]=train[prompt_col].map(detect_category)
print(train["category"].value_counts().to_string())
parts=[]
for cat,n in SAMPLE_PLAN.items():
    sub=train[train.category==cat]
    if len(sub)<n: raise ValueError(f"Requested {n} rows for {cat}, found {len(sub)}")
    parts.append(sub.sample(n=n,random_state=RANDOM_SEED))
sample=pd.concat(parts).sample(frac=1,random_state=RANDOM_SEED).reset_index(drop=True)
records=[]
for _,row in sample.iterrows():
    prompt=str(row[prompt_col]); ans=str(row[answer_col]).strip(); cat=str(row["category"]); ast=assistant(cat,prompt,ans)
    records.append({"id":str(row[id_col]) if id_col else str(row.name),"category":cat,"prompt":prompt,"answer":ans,"messages":[{"role":"user","content":prompt.rstrip()+"\nPlease put your final answer inside `\\boxed{}`."},{"role":"assistant","content":ast}],"assistant":ast})
raw_jsonl=WORKING_DIR/RAW_JSONL_NAME; raw_sum=WORKING_DIR/RAW_SUMMARY_NAME; raw_zip=WORKING_DIR/RAW_ZIP_NAME
write_jsonl(records,raw_jsonl); raw_summary=write_summary(records,raw_sum); write_zip([raw_jsonl,raw_sum],raw_zip)
qa=[]; clean=[]
for r in records:
    ok,reason=is_safe(r); qa.append({"id":r["id"],"category":r["category"],"reason":reason})
    if ok: clean.append(r)
qa_df=pd.DataFrame(qa); print("Metric-safe QA summary:"); print(qa_df.groupby(["category","reason"]).size().reset_index(name="count").to_string(index=False))
bad=qa_df[qa_df.reason!="ok"]
if len(bad): print("Dropped metric-unsafe rows:\n"+bad.to_string(index=False))
clean_jsonl=WORKING_DIR/CLEAN_JSONL_NAME; clean_sum=WORKING_DIR/CLEAN_SUMMARY_NAME; clean_zip=WORKING_DIR/CLEAN_ZIP_NAME
write_jsonl(clean,clean_jsonl); clean_summary=write_summary(clean,clean_sum); write_zip([clean_jsonl,clean_sum],clean_zip)
assert len(records)==sum(SAMPLE_PLAN.values())
assert len(clean)>=700
assert not any("Ignore story text" in r["assistant"] for r in clean)
assert len({r["id"] for r in clean})==len(clean)
for r in clean:
    assert extract_final_answer(r["assistant"])==r["answer"], r
    assert r["assistant"].count("\\boxed{")==1, r
    if r["category"]=="bit_manipulation": assert re.fullmatch(r"[01]{8}",r["answer"]), r
print("Wrote raw:",raw_jsonl,raw_sum,raw_zip)
print("Wrote clean:",clean_jsonl,clean_sum,clean_zip)
print(clean_summary.to_string(index=False))
eq_count=Counter(r["category"] for r in clean).get("equation_numeric_deduce",0)
print("PASS: Notebook B TRACE_JSONL_NAME =",CLEAN_JSONL_NAME)
print(f"PASS: Notebook B OUTPUT_TAG = adapter_sft_v3_mix_bitfix_cipher_eq{eq_count}_clean")

category
bit_manipulation           1602
gravity                    1597
unit_conversion            1594
numeral                    1576
cipher                     1576
cryptarithm_guess           823
equation_numeric_deduce     732
Metric-safe QA summary:
               category reason  count
       bit_manipulation     ok    256
                 cipher     ok    192
equation_numeric_deduce     ok     96
                gravity     ok     64
                numeral     ok     64
        unit_conversion     ok     64
Wrote raw: /kaggle/working/train_traces_v3_mix.jsonl /kaggle/working/train_traces_v3_mix_summary.csv /kaggle/working/train_traces_v3_mix.zip
Wrote clean: /kaggle/working/train_traces_v3_mix_clean_metric_safe.jsonl /kaggle/working/train_traces_v3_mix_clean_metric_safe_summary.csv /kaggle/working/train_traces_v3_mix_clean_metric_safe.zip
               category  count
       bit_manipulation    256
                 cipher    192
equation_numeric_deduce     96
               